In [ ]:
import os
import numpy as np
import xarray as xr

%matplotlib inline
import matplotlib.pyplot as plt
from matplotlib import colors
from matplotlib.patches import Rectangle
from matplotlib.tri import Triangulation

import cartopy.crs as ccrs
import cartopy.feature as cfeature

In [ ]:
grid = xr.open_dataset("./grids/icon_grid_0026_R03B07_G.nc")
grid = grid.rename_dims({"cell": "values"})

tri = Triangulation(np.rad2deg(grid["clon"]), np.rad2deg(grid["clat"]))

lsm = xr.open_dataset("./grids/icon_extpar_0026_R03B07_G_20140731.nc")["FR_LAND"].values > 0.99

In [ ]:
# Define "study area"
x0, y0 = -5, 37
wx, wy = 30, 25

cells = ((np.rad2deg(grid["clon"]) >= x0) & (np.rad2deg(grid["clon"]) <= (x0+wx)) & 
         (np.rad2deg(grid["clat"]) >= y0) & (np.rad2deg(grid["clat"]) <= (y0+wy))).values

mask = lsm & cells

area_weights = grid.isel(values=mask)["cell_area"]
area_weights.name = "weights"

In [ ]:
def load_part(fname, mask):
    ds = xr.open_dataset(fname)
    if "step" in ds.dims:
        return ds.sel(values=mask).isel(step=2)["W_SO"] #depthBelowLandLayer=layer, 
    else:
         return ds.sel(values=mask)["W_SO"] #depthBelowLandLayer=layer, 

In [ ]:
path = "./data/"
fnames = [fname for fname in os.listdir(path)]
fnames = [fname for fname in fnames if fname.endswith(".grb")]
fnames = [path + fname for fname in fnames if fname.startswith("sm_")]

In [ ]:
results = []
for fname in fnames:
    #ds = xr.open_dataset(fname)
    #break
    ds = load_part(fname, mask)
    ds_weighted = ds.weighted(area_weights)
    results.append(ds_weighted.mean(dim="values"))

In [ ]:
ds_mean = xr.concat(results, dim="time")

In [ ]:
ds_mean

In [ ]:
for depth in ds_mean["depthBelowLandLayer"].values:
    plt.plot(ds_mean["valid_time"], ds_mean.sel(depthBelowLandLayer=depth), label=f"{depth:.2f}")
    plt.legend()
    plt.xticks(rotation=45)
    plt.show()
    

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10,4), subplot_kw={'projection': ccrs.PlateCarree()})
fig.tight_layout()

ax.set_extent([-5, 25, 37, 62], crs=ccrs.PlateCarree())

ax.add_feature(cfeature.LAND)
ax.add_feature(cfeature.OCEAN)
ax.add_feature(cfeature.COASTLINE)
ax.add_feature(cfeature.BORDERS, linestyle=':')

im = ax.tricontourf(tri, ds_sm.sel(depthBelowLandLayer=0.0, time="2021-03-01T21"))#, levels=lvls, norm=norm)
#im = ax.contourf(dr_out["lon"], dr_out["lat"], 
#                 dr_out.sel(time="2021-07-15T00").data - dr_out.sel(time="2021-07-13T00").data, 
#                 levels=lvls, norm=norm)

# Create a Rectangle patch
#rectangle = Rectangle((x0, y0), wx, wy, edgecolor='red', linewidth=2, fill=False)
#ax.add_patch(rectangle)

#plt.title("ICON Factual Deterministic Run")
plt.colorbar(im, label="48h total Precipitation in kg / m^2")#, ticks=lvls)
#plt.savefig('./figs/icon_fdet.png', dpi=300, bbox_inches='tight', format='png')
plt.show()


#mask_out = (dr_out["lon"] > x0) & (dr_out["lon"] < (x0+wx)) & (dr_out["lat"] > y0) & (dr_out["lat"] < (y0+wy))
#prec48_tsmp = calc_spatial_integral(dr_out.sum(dim="time").where(mask_out), "lon", "lat")
#print(f"48h area precipitation sum: {prec48_tsmp:.3e} mm")

#prec48_fdet = ((ds_fdet_prec.sel(time="2021-07-15T00") - ds_fdet_prec.sel(time="2021-07-13T00")) * icon_area.data).isel(ncells_2=cells).sum().item()
#print(f"48h precipitation sum: {prec48_fdet:.3e} mm")